# Predict With Trained YOLO-lite Checkpoint

Use this notebook locally after downloading `best.pth` from Kaggle. Put the checkpoint path and image folder in the settings cell, then run all cells.

## 1. Settings

In [22]:
from pathlib import Path
import sys

# Project root. If this notebook is inside the project folder, keep this as current directory.
PROJECT_DIR = Path.cwd()

# Download best.pth from Kaggle and set the path here.
CHECKPOINT = PROJECT_DIR / "output"/ "v4" / "best.pth"
IMAGE_DIR = PROJECT_DIR / "public" / "val" / "images"

# Output JSON path.
OUTPUT_JSON = PROJECT_DIR /"output"/ "predictions.json"

# Inference settings.
BATCH_SIZE = 32
CONF_THRESHOLD = 0.20 # lọc các box dự đoán, confidence score trên x sẽ giữ, tăng lên sẽ tăng prediction giảm recall
NMS_THRESHOLD = 0.50 # loại các box chồng nhau
MAX_DETECTIONS = 100
DEVICE = "cuda" 
# Optional validation scoring. Keep these paths if evaluating public val.
GROUND_TRUTH_JSON = PROJECT_DIR / "public" / "annotations" / "val.json"
EVALUATOR = PROJECT_DIR / "public" / "tools" / "evaluate_predictions.py"
SCORE_JSON = PROJECT_DIR /"output"/ "score.json"


## 3. Run Prediction

In [23]:
print("Predict command:")
! python predict.py --image_dir "{IMAGE_DIR}" --output "{OUTPUT_JSON}" --checkpoint "{CHECKPOINT}" --batch_size {BATCH_SIZE} --conf_threshold {CONF_THRESHOLD} --nms_threshold {NMS_THRESHOLD} --max_detections {MAX_DETECTIONS} --device {DEVICE}


Predict command:
wrote 1500 predictions to e:\Duong\onedrive\OneDrive - k52a0\CN12-IAI\Third_year\Image processing\object detection\output\predictions.json


## 5. Optional Evaluate On Validation

In [24]:
import json
import subprocess
import sys
if GROUND_TRUTH_JSON.exists() and EVALUATOR.exists():
    print("Ground truth:", GROUND_TRUTH_JSON)
    print("Evaluator:", EVALUATOR)
    print("Evaluate command:")
    print(f"{sys.executable} {EVALUATOR} --ground_truth {GROUND_TRUTH_JSON} --predictions {OUTPUT_JSON} --output {SCORE_JSON}")
    subprocess.run(
    [
        sys.executable,
        EVALUATOR,
        "--ground_truth", GROUND_TRUTH_JSON,
        "--predictions", OUTPUT_JSON,
        "--output", SCORE_JSON
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
    score = json.loads(SCORE_JSON.read_text(encoding="utf-8"))

    summary_rows = [
        {"metric": "mAP@0.5", "value": score.get("mAP@0.5")},
        {"metric": "performance_points", "value": score.get("performance_points")},
        {"metric": "micro_precision", "value": score.get("micro_precision")},
        {"metric": "micro_recall", "value": score.get("micro_recall")},
        {"metric": "num_ground_truth_boxes", "value": score.get("num_ground_truth_boxes")},
        {"metric": "num_predictions", "value": score.get("num_predictions")},
    ]
    print("\nEvaluation summary")
    try:
        import pandas as pd
        display(pd.DataFrame(summary_rows))
        if "per_class" in score:
            per_class_df = pd.DataFrame.from_dict(score["per_class"], orient="index")
            per_class_df = per_class_df[[
                "ap", "recall", "precision", "num_ground_truth",
                "num_predictions", "true_positives", "false_positives",
            ]]
            per_class_df = per_class_df.sort_values("ap", ascending=False)
            print("\nPer-class metrics")
            display(per_class_df)
    except Exception:
        for row in summary_rows:
            print(f"{row['metric']}: {row['value']}")
        if "per_class" in score:
            print("\nPer-class metrics")
            for class_name, metrics in score["per_class"].items():
                print(
                    f"{class_name}: AP={metrics.get('ap')} "
                    f"recall={metrics.get('recall')} "
                    f"precision={metrics.get('precision')} "
                    f"GT={metrics.get('num_ground_truth')} "
                    f"pred={metrics.get('num_predictions')}"
                )
else:
    print("Skipped evaluation, so mAP@0.5 cannot be shown.")
    print("mAP@0.5 is only available when you predict on a labeled split, e.g. public/val/images.")
    print("GROUND_TRUTH_JSON:", GROUND_TRUTH_JSON, GROUND_TRUTH_JSON.exists())
    print("EVALUATOR:", EVALUATOR, EVALUATOR.exists())


Ground truth: e:\Duong\onedrive\OneDrive - k52a0\CN12-IAI\Third_year\Image processing\object detection\public\annotations\val.json
Evaluator: e:\Duong\onedrive\OneDrive - k52a0\CN12-IAI\Third_year\Image processing\object detection\public\tools\evaluate_predictions.py
Evaluate command:
c:\Users\hoang\.conda\envs\objdetection\python.exe e:\Duong\onedrive\OneDrive - k52a0\CN12-IAI\Third_year\Image processing\object detection\public\tools\evaluate_predictions.py --ground_truth e:\Duong\onedrive\OneDrive - k52a0\CN12-IAI\Third_year\Image processing\object detection\public\annotations\val.json --predictions e:\Duong\onedrive\OneDrive - k52a0\CN12-IAI\Third_year\Image processing\object detection\output\predictions.json --output e:\Duong\onedrive\OneDrive - k52a0\CN12-IAI\Third_year\Image processing\object detection\output\score.json

Evaluation summary


,metric,value
0,mAP@0.5,0.582174
1,performance_points,20.000000
2,micro_precision,0.756682
3,micro_recall,0.672439
4,num_ground_truth_boxes,2021.000000
5,num_predictions,1796.000000



Per-class metrics


,ap,recall,precision,num_ground_truth,num_predictions,true_positives,false_positives
dog,0.703692,0.742718,0.739130,206,207,153,54
person,0.696835,0.748603,0.789009,1074,1019,804,215
cat,0.664687,0.681818,0.895522,176,134,120,14
car,0.529918,0.565371,0.772947,283,207,160,47
chair,0.315738,0.432624,0.532751,282,229,122,107


## 6. Threshold Sweep On Validation

Run this only when `IMAGE_DIR` is validation images and `GROUND_TRUTH_JSON` is available. It helps choose better confidence/NMS thresholds without retraining.

In [ ]:
RUN_THRESHOLD_SWEEP = False
CONF_VALUES = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
NMS_VALUES = [0.45, 0.50, 0.60]

if RUN_THRESHOLD_SWEEP:
    if not (GROUND_TRUTH_JSON.exists() and EVALUATOR.exists()):
        raise FileNotFoundError("Need GROUND_TRUTH_JSON and EVALUATOR for threshold sweep.")
    rows = []
    for conf in CONF_VALUES:
        for nms in NMS_VALUES:
            sweep_pred = PROJECT_DIR /"output"/ f"predictions_conf{conf:.2f}_nms{nms:.2f}.json"
            sweep_score = PROJECT_DIR /"output"/ f"score_conf{conf:.2f}_nms{nms:.2f}.json"
            print(f"Running conf={conf:.2f} nms={nms:.2f}")
            !{sys.executable} predict.py --image_dir "{IMAGE_DIR}" --output "{sweep_pred}" --checkpoint "{CHECKPOINT}" --batch_size {BATCH_SIZE} --conf_threshold {conf} --nms_threshold {nms} --max_detections {MAX_DETECTIONS} --device {DEVICE}
            !{sys.executable} "{EVALUATOR}" --ground_truth "{GROUND_TRUTH_JSON}" --predictions "{sweep_pred}" --output "{sweep_score}"
            score = json.loads(sweep_score.read_text(encoding="utf-8"))
            rows.append({
                "conf": conf,
                "nms": nms,
                "map50": score.get("mAP@0.5", 0.0),
                "precision": score.get("micro_precision", 0.0),
                "recall": score.get("micro_recall", 0.0),
                "num_predictions": score.get("num_predictions", 0),
            })
    try:
        import pandas as pd
        display(pd.DataFrame(rows).sort_values("map50", ascending=False))
    except Exception:
        print(json.dumps(rows, indent=2))
else:
    print("Threshold sweep disabled. Set RUN_THRESHOLD_SWEEP = True to run it.")


Running conf=0.05 nms=0.45
wrote 1500 predictions to e:\Duong\onedrive\OneDrive - k52a0\CN12-IAI\Third_year\Image processing\object detection\predictions_conf0.05_nms0.45.json
{
  "mAP@0.5": 0.6677,
  "performance_points": 20,
  "iou_threshold": 0.5,
  "num_ground_truth_boxes": 2021,
  "num_predictions": 10829,
  "micro_precision": 0.159664,
  "micro_recall": 0.855517,
  "per_class": {
    "person": {
      "ap": 0.76564,
      "num_ground_truth": 1074,
      "num_predictions": 5429,
      "true_positives": 963,
      "false_positives": 4466,
      "recall": 0.896648,
      "precision": 0.177381
    },
    "car": {
      "ap": 0.605667,
      "num_ground_truth": 283,
      "num_predictions": 2036,
      "true_positives": 236,
      "false_positives": 1800,
      "recall": 0.833922,
      "precision": 0.115914
    },
    "dog": {
      "ap": 0.762595,
      "num_ground_truth": 206,
      "num_predictions": 582,
      "true_positives": 175,
      "false_positives": 407,
      "recall": 0

,conf,nms,map50,precision,recall,num_predictions
0,0.05,0.45,0.667700,0.159664,0.855517,10829
1,0.05,0.50,0.665844,0.137917,0.866403,12696
2,0.05,0.60,0.652380,0.097798,0.881247,18211
3,0.10,0.45,0.649281,0.398199,0.787729,3998
4,0.10,0.50,0.649257,0.364541,0.797625,4422
5,0.10,0.60,0.638557,0.287271,0.808511,5688
6,0.15,0.45,0.616454,0.606111,0.726373,2422
7,0.15,0.50,0.614337,0.575497,0.729837,2563
8,0.15,0.60,0.608620,0.498334,0.740228,3002
9,0.20,0.45,0.568730,0.746807,0.665512,1801
